# Filtro IIR Butterworth: de pseudocódigo a Python

Se genera una señal IQ sintética `X` de forma `(N, 2, L)`, se filtra su eje temporal y se compara la potencia antes y después. El índice `0` contiene I y el índice `1` contiene Q.

## Pseudocódigo

```text
FUNCTION GenerateSyntheticIQ(exampleCount, sampleCount, seed)
    DECLARE rng, sampleIndex, I, Q, iqArray AS ARRAY
    rng ← RandomGenerator(seed)
    sampleIndex ← ArrayFromZeroTo(sampleCount - 1)
    I ← Sinusoid(0.03, sampleIndex) + 0.70 × RandomNormal(rng, exampleCount, sampleCount)
    Q ← Cosine(0.03, sampleIndex) + 0.70 × RandomNormal(rng, exampleCount, sampleCount)
    iqArray ← Stack(I, Q, axis=1)
    RETURN iqArray
END FUNCTION

FUNCTION DesignButterworthIIR(cutoff, sampleRate, order)
    DECLARE b, a AS ARRAY
    (b, a) ← ButterworthLowPass(order, cutoff, sampleRate)
    RETURN (b, a)
END FUNCTION

FUNCTION ApplyIIRFilter(signal, b, a)
    DECLARE filtered AS ARRAY
    filtered ← IIRFilter(b, a, signal, axis=2)
    RETURN filtered
END FUNCTION

FUNCTION ComputePower(iqArray)
    DECLARE iComponent, qComponent, powerPerExample AS ARRAY
    iComponent ← iqArray[:, 0, :]
    qComponent ← iqArray[:, 1, :]
    powerPerExample ← Mean(iComponent² + qComponent², axis=1)
    RETURN powerPerExample
END FUNCTION

--- Main ---
N ← 5
L ← 1000
seed ← 42
X ← GenerateSyntheticIQ(N, L, seed)
(b, a) ← DesignButterworthIIR(cutoff=0.10, sampleRate=1.0, order=2)
XFiltered ← ApplyIIRFilter(X, b, a)
powerBefore ← ComputePower(X)
powerAfter ← ComputePower(XFiltered)
```

In [ ]:
import numpy as np
from scipy import signal


def generate_synthetic_iq(example_count: int, sample_count: int, seed: int) -> np.ndarray:
    """Return float32 IQ data with shape (N, 2, L)."""
    rng = np.random.default_rng(seed)
    sample_index = np.arange(sample_count, dtype=np.float32)
    phase = 2 * np.pi * 0.03 * sample_index
    i_component = np.sin(phase)[None, :] + 0.70 * rng.standard_normal((example_count, sample_count))
    q_component = np.cos(phase)[None, :] + 0.70 * rng.standard_normal((example_count, sample_count))
    return np.stack((i_component, q_component), axis=1).astype(np.float32)


def design_butterworth_iir(cutoff: float, sample_rate: float, order: int) -> tuple[np.ndarray, np.ndarray]:
    return signal.butter(order, cutoff, btype="lowpass", fs=sample_rate)


def apply_iir_filter(iq_array: np.ndarray, b: np.ndarray, a: np.ndarray) -> np.ndarray:
    return signal.lfilter(b, a, iq_array, axis=2)


def compute_power(iq_array: np.ndarray) -> np.ndarray:
    i_component = iq_array[:, 0, :]
    q_component = iq_array[:, 1, :]
    return np.mean(i_component**2 + q_component**2, axis=1)

In [ ]:
N, L, SEED = 5, 1_000, 42
cutoff, sample_rate, order = 0.10, 1.0, 2

X = generate_synthetic_iq(N, L, SEED)
b, a = design_butterworth_iir(cutoff, sample_rate, order)
X_filtered = apply_iir_filter(X, b, a)
power_before = compute_power(X)
power_after = compute_power(X_filtered)

print(f"Input shape:  {X.shape}")
print(f"Output shape: {X_filtered.shape}")
print(f"Butterworth numerator (b): {b}")
print(f"Butterworth denominator (a): {a}")
print(f"Mean power before: {power_before.mean():.4f}")
print(f"Mean power after:  {power_after.mean():.4f}")
print(f"Power change:      {(power_after.mean() - power_before.mean()):.4f}")

assert X.shape == (N, 2, L)
assert X_filtered.shape == X.shape
assert not np.allclose(power_before, power_after), "Filtering should change the power"
assert power_after.mean() < power_before.mean(), "Low-pass filtering should reduce the injected high-frequency noise power"
print("PASS: the 2nd-order IIR filter preserves shape and changes power.")